# Lab 6 — Optimization: Build a Local Model, Step to Its Minimum

**ROB7104 / ROB8104 Mathematics for Robotics · gradient descent, Newton's method, noise, sparsity, paths and manifolds**

This notebook is a reading. There is no code to run and nothing to hand in. Every idea comes with a short video of a simulated robot arm, and every video has a live version on the lab's web page where you can turn the camera and move the sliders yourself:

**Interactive page:** [realgaule.github.io/rob7104/lab6](https://realgaule.github.io/rob7104/lab6/)

It follows Lecture 4 (optimization). The Gauss–Newton step also appears in Lecture 13, and Lagrange multipliers in Lecture 5.

> If a video does not play inside your notebook viewer, open `Lab6.html` (same folder) in a browser.

## 0. One arm, one target: reaching is minimizing

<video controls loop muted playsinline width="100%" src="media/s00_opening.mp4" poster="media/s00_opening_poster.png"></video>

*Four copies of the same arm, the same start and the same target, four methods. Below each arm, its cost landscape; the white point on the map is the arm above it.*

**The task.** A six-joint arm should put its tool point $p(q)$ on a target $t$. Written as an optimization problem:

$$
\min_q\ f(q), \qquad f(q) = \tfrac12\,\lVert p(q) - t\rVert^2 .
$$

The gradient is $\nabla f(q) = J(q)^\top\big(p(q) - t\big)$, where $J = \partial p/\partial q$ is the Jacobian of the arm.

**A landscape you can see.** Most of the lab freezes four joints and lets only the shoulder and the elbow turn. The arm then moves in a vertical plane like a two-link arm (links 0.425 m and 0.492 m, reach 0.067 to 0.917 m from the shoulder axis), and the cost is a function of two angles $(q_1, q_2)$. Draw it as a map: **every point of the map is one pose of the arm**, and an optimizer walking on the map is the arm moving. Because angles are periodic, the map wraps around at its edges. For a reachable target it has six stationary points ($\nabla f = 0$):

| Stationary point | Pose of the arm | Type |
|---|---|---|
| elbow up (green) | tool on the target | minimum, $f = 0$ |
| elbow down (blue) | tool on the target | minimum, $f = 0$ |
| stretched towards the target | arm straight, pointing at the target | saddle (yellow) |
| two folded poses | forearm folded back onto the upper arm | saddles (yellow) |
| stretched away from the target | arm straight, pointing away | maximum (red) |

**What the video shows.** The target is at 0.9 of the reach, and all four arms start from the same pose.

| Method | What the arm does |
|---|---|
| Gradient descent, large step | zig-zags across the valley and settles: within 1 mm of the target after 64 steps |
| Gradient descent with a noisy gradient | trembles into the same solution and keeps trembling (2.8 cm root-mean-square miss) |
| Safeguarded Newton | within 1 mm after 5 steps |
| Newton's method | 3 confident steps to the **maximum**: arm straight, tool 1.74 m from the target |

The fastest method also gives the worst answer. This lab explains all four behaviours.

### 0.1 The one idea of this lab

Lecture 4 sums up its methods in one sentence: **at the current point, build a local model of the cost, then step to the minimum of that model. The model has to fit the structure of the problem.** Each section is one choice of model:

| What we know about the problem | Local model | Method | Section |
|---|---|---|---|
| only the slope | tangent plane plus a penalty on the step length | gradient descent | 1.1 |
| the curvature | quadratic surface | Newton's method | 1.2 |
| the cost is a sum of squared residuals | linearised residual | Gauss–Newton, damped least squares | 1.3 |
| the gradient is noisy | slope plus a random term | noisy gradient descent | 2.1 |
| the cost has a kink | linearise the smooth part, keep the kink | proximal gradient | 2.2 |
| the unknown is a whole curve | variation of the curve | Euler–Lagrange equation | 3.1 |
| the unknown lives on a curved surface | step in the tangent space, then pull back | optimization on manifolds | 3.2 |

Links to earlier labs: least squares and the SVD (Lab 1) return in 1.3; Langevin dynamics and the density $e^{-U/T}$ (Lab 3) return in 2.1.

**A word on what you are watching.** The arm is moved kinematically: each frame sets its joint angles to the current iterate of the optimizer (interpolated between iterates for display). It is not a simulation of forces, friction or contact. The arm model is the UR5e of MuJoCo Menagerie. Joint limits are ignored and angles are periodic. Every number quoted below was computed by the scripts that made the videos.

---

# Part 1 — Slope and curvature

## 1.1 Gradient descent and the step size

**The local model.** Gradient descent trusts only the slope. It models the cost by its tangent plane and adds a penalty for going far:

$$
q_{k+1} = \arg\min_q\ \Big[f(q_k) + \nabla f(q_k)^\top (q - q_k) + \tfrac{1}{2\alpha}\lVert q - q_k\rVert^2\Big] = q_k - \alpha\,\nabla f(q_k).
$$

**What the step size does.** Near a minimum $q^\ast$ the cost is close to a quadratic, $f \approx \tfrac12 (q-q^\ast)^\top H\,(q-q^\ast)$ with Hessian $H$. Write the error $q_k - q^\ast$ in the eigenvectors of $H$. Along eigenvector $i$ (eigenvalue $\lambda_i$) one step multiplies the error by

$$
1 - \alpha\lambda_i .
$$

So gradient descent converges near $q^\ast$ exactly when $\lvert 1 - \alpha\lambda_i\rvert < 1$ for every $i$, that is, when $0 < \alpha < 2/\lambda_{\max}$. Lecture 4's example $f = \tfrac12(x_1^2 + 9x_2^2)$ has the limit $2/9$; at $\alpha = 0.25$ the second component is multiplied by $-1.25$ at every step and blows up while flipping sign.

**On the arm.** Where the tool reaches the target, the Hessian is $H = J^\top J$ (the other term of the Hessian is multiplied by the residual $p - t$, which is zero there). For the default target its eigenvalues are $\lambda_1 = 0.082$ (flat direction) and $\lambda_2 = 0.523$ (steep direction). The eigenvectors are the right singular vectors of $J$: turning the joints along the steep direction moves the tool 0.72 m per radian, along the flat direction only 0.29 m per radian.

<video controls loop muted playsinline width="100%" src="media/s11a_stepsize.mp4" poster="media/s11a_stepsize_poster.png"></video>

*Left: the arm and the trail of its tool. Top right: the paths on the landscape, with the steep (solid) and flat (dashed) directions. Bottom right: the factor $\lvert 1 - \alpha\lambda_i\rvert$ for both directions against the step size; above 1 the error grows.*

| Step size | Factor (flat) | Factor (steep) | What the arm does |
|---|---|---|---|
| small, $\alpha = 0.38$ | 0.97 | 0.80 | creeps; within 1 mm after 181 steps |
| near the limit, $\alpha = 3.64$ | 0.70 | $-0.90$ | swings across the valley, the steep error flips sign every step; within 1 mm after 52 steps |
| beyond the limit, $\alpha = 4.31$ | 0.65 | $-1.25$ | started 1 mm from the target, swings wider every step; more than 10 cm away after 22 steps |

Two things the linear picture does not tell you:

- **Beyond the limit the arm does not fly off.** The joint angles wrap around and the gradient of a reaching arm is bounded, so the iterates cannot run off to infinity. From all of 2000 random starts the run ends in a permanent swing between two poses (tool 23 cm and 30 cm from the target). The local factor predicts that the solution repels; where the arm ends up is a nonlinear question.
- **Local and global limits differ.** $2/\lambda_2 = 3.83$ is the limit near the solution. The step that is safe from every start uses the largest curvature over the whole map, $L = 0.826$, so $2/L = 2.42$. With $\alpha = 1/L$ gradient descent reaches a minimum from 100 % of starts (median 178 steps to $\lVert\nabla f\rVert \le 10^{-9}$); near the solution its error shrinks by $1 - \lambda_1/L = 0.90$ per step.

### Ill-conditioning is a pose of the robot

The condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ of $H = J^\top J$ is the squared ratio of the singular values of $J$. Geometrically, it is how flat the **manipulability ellipse** at the tool is: the set of tool velocities that unit joint speeds can produce. The flat valley of the landscape and the flat ellipse are the same thing. Moving the target out towards the edge of the workspace stretches the arm and flattens both:

| Target distance (fraction of reach) | 0.5 | 0.66 | 0.8 | 0.9 | 0.99 | 0.994 |
|---|---|---|---|---|---|---|
| condition number $\kappa$ | 4.1 | 6.4 | 13 | 30 | 330 | 552 |
| gradient descent, $\alpha = 1/L$: steps to 0.1 mm | 46 | 72 | 126 | 243 | 1686 | 2500 |
| heavy ball (momentum): steps to 0.1 mm | 9 | 11 | 18 | 33 | 127 | 159 |

**Momentum.** The heavy-ball method adds a fraction of the previous step, $q_{k+1} = q_k - \alpha\nabla f(q_k) + \beta\,(q_k - q_{k-1})$. Across the valley successive steps point in opposite directions and cancel; along the valley they point the same way and add up. With Polyak's choice of $\alpha$ and $\beta$ (tuned to the curvatures at the solution) it needs 159 instead of 2500 steps at 0.994 of the reach.

> **Takeaway.** The steepest curvature limits the step size, the flattest curvature sets the speed; their ratio is the condition number, and for an arm it is a pose.
> ▶ Try it: [Gradient descent and step size](https://realgaule.github.io/rob7104/lab6/#d11): slide the step size past $2/L$, switch on momentum, move the target out.

## 1.2 Newton's method and curvature

**The local model.** Newton's method also uses the curvature. Its model is the quadratic

$$
m_k(d) = f(q_k) + \nabla f(q_k)^\top d + \tfrac12\, d^\top H(q_k)\, d ,
\qquad
q_{k+1} = q_k - H(q_k)^{-1}\nabla f(q_k).
$$

The step goes to the stationary point of the model. If $H$ is positive definite the model is a **bowl** and that point is its bottom. On a strictly convex quadratic one Newton step lands exactly on the minimum (Lecture 4's example reaches $(1,1)$ in one step). But if $H$ is indefinite the model is a **saddle**, and if $H$ is negative definite it is a **dome**, and Newton's method goes to the top of the dome just as willingly. **Newton's method looks for stationary points, not for minima.**

On the arm's landscape the Hessian is positive definite on only 14 % of the map, negative definite on 23 % and indefinite on 62 %.

<video controls loop muted playsinline width="100%" src="media/s12_newton.mp4" poster="media/s12_newton_poster.png"></video>

*Three starts. Pink: Newton's method; white: safeguarded Newton; the local model at each step is drawn as a green bowl, a yellow saddle or a red dome. Then the basin map: each start coloured by where Newton's method ends.*

| Start | Local models on the way | Newton's method ends at | Steps |
|---|---|---|---|
| A | bowls | the elbow-up minimum | 4 (gradient descent: 176) |
| B | domes | the **maximum**: arm stretched away, tool 1.52 m from the target | 4 |
| C | saddles | the saddle "stretched towards the target", tool 0.31 m from the target | 4 |

The step counts are to $\lVert\nabla f\rVert \le 10^{-9}$; to within 1 mm of the target, start A takes 3 Newton steps against 56 gradient steps. When Newton's method works, it is fast: near a minimum the number of correct digits roughly doubles with every step.

**The basin map.** Run pure Newton from every start on the map and colour the start by where it ends. Only about 20 % of the starts end at a minimum (10 % at each), 59 % at a saddle and 21 % at the maximum (200 000 starts). The map is fractured: moving each start by $10^{-7}$ rad changes the end point for 2.1 % of them.

**The safeguard (Lecture 4).** Where $H$ is not positive definite, replace it by $H + \tau I$ with $\tau$ just large enough to make it a bowl; here the smallest eigenvalue is raised to $\max(\delta, \lvert\lambda_{\min}\rvert)$ with $\delta = 10^{-3}$. Then shorten the step until the cost decreases enough (Armijo backtracking). The direction is now always downhill. Safeguarded Newton reaches a minimum from 100 % of the starts, half at each, with a median of 6 steps; from start B it reaches the elbow-down minimum in 6 steps. Gradient descent with $\alpha = 1/L$ also reaches a minimum from every start, but needs a median of 178 steps.

> **Takeaway.** A quadratic model is a good model only when it is a bowl. Newton's method is fast near a minimum and goes to the wrong place elsewhere; shifting the curvature keeps the speed and removes the danger.
> ▶ Try it: [Newton's method and curvature](https://realgaule.github.io/rob7104/lab6/#d12): click on the landscape to start, step, and switch the safeguard on and off.

## 1.3 Inverse kinematics is optimization

Now all six joints move, and the tool must reach a **position and an orientation**. Stack the position error and the orientation error (weighted by 0.3 m per radian) into one residual $e(q) \in \mathbb{R}^6$, with Jacobian $J = \partial e/\partial q$, and minimise $\tfrac12\lVert e(q)\rVert^2$. Three local models give the three classical solvers:

| Solver | Local model | Step $\Delta q$ |
|---|---|---|
| Jacobian transpose | slope only (gradient descent) | $-\alpha\, J^\top e$ |
| Gauss–Newton | linearised residual $e + J\Delta q$, solved by least squares | $-J^{+} e$ |
| Damped least squares | linearised residual plus a penalty $\lambda^2\lVert\Delta q\rVert^2$ | $-(J^\top J + \lambda^2 I)^{-1} J^\top e$ |

Gauss–Newton is Newton's method with the Hessian replaced by $J^\top J$: it drops the term that is multiplied by the residual, which is exact where the residual is zero (as in 1.1). Its step is the least-squares solution of Lab 1.

**Through the SVD.** Write $J = \sum_i \sigma_i u_i v_i^\top$. Each solver moves the joints along $v_i$ by a gain times the component $u_i^\top e$ of the error:

$$
\text{gain}(\sigma) = \alpha\,\sigma \ \ (\text{transpose}),\qquad
\frac{1}{\sigma}\ \ (\text{Gauss–Newton}),\qquad
\frac{\sigma}{\sigma^2 + \lambda^2} \le \frac{1}{2\lambda}\ \ (\text{damped}).
$$

When the arm approaches a singular pose, some $\sigma_i \to 0$. Gauss–Newton's gain $1/\sigma$ explodes; the damped gain stays below $1/(2\lambda)$ and falls back to zero.

<video controls loop muted playsinline width="100%" src="media/s13_ik.mp4" poster="media/s13_ik_poster.png"></video>

*The target pose moves around a loop; about a third of the loop (red) lies beyond the reach of the arm, up to about 10 cm. One solver iteration per frame. Right: the residual and the joint step per iteration, and the gain of each solver against the singular value.*

| | Target at rest: iterations to 1 mm | Target out of reach |
|---|---|---|
| Jacobian transpose | 32 | lags behind (24 mm off after the 8 resting iterations); 13 cm from the target at the far point |
| Gauss–Newton | 3 | thrashes: 25 joint steps above 1 rad, the largest 61 rad; calm again only once the target is reachable |
| Damped, $\lambda = 0.10$ | 3 | rests at the closest pose, 9.9 cm from the target; no restart from 60 random starts finds a closer one |

Midway, the clip freezes one Gauss–Newton step. It is 1.83 rad long, 99.9 % of it along the weakest singular direction ($\sigma_6 = 0.038$), and it takes the tool from 8.2 cm to 22.2 cm away. Replaying the same step with growing damping turns it towards the gradient direction and shortens it; at $\lambda = 0.3$ it is 0.049 rad long and leaves the tool 6.9 cm away.

**Damping is not a cure-all.** Out of reach, a fixed damping works only if it is large enough: here above about $\sqrt{h/2} \approx 0.08$, where $h$ is the curvature of the cost along the lost direction. With $\lambda = 0.05$ the arm hops by about 1.1 rad at every iteration for as long as the target stays out of reach. The price of damping is speed: at rest it needs 11 iterations to $10^{-6}$ where Gauss–Newton needs 4.

> **Takeaway.** Inverse kinematics is least squares on a residual. Gauss–Newton is fast while the target is reachable; damping trades a little speed for steps that stay bounded near singular poses.
> ▶ Try it: [Inverse kinematics](https://realgaule.github.io/rob7104/lab6/#d13): drag the target out of reach and switch between the three solvers.

---

# Part 2 — Noise and kinks

Part 1 assumed a smooth cost and an exact gradient. Here we drop one assumption at a time.

## 2.1 Gradient descent with noise

**The update.** In practice the gradient is often noisy (estimated from samples, measured by a sensor). Model this as

$$
q_{k+1} = q_k - \alpha\big(\nabla f(q_k) + \sigma_g\,\xi_k\big), \qquad \xi_k \sim N(0, I).
$$

**Noise is not only a nuisance.** At the maximum and at the saddles the gradient is exactly zero, so plain gradient descent started there never moves (in exact arithmetic). Noise pushes the iterate off, and the negative curvature then carries it away.

<video controls loop muted playsinline width="100%" src="media/s21a_noise.mp4" poster="media/s21a_noise_poster.png"></video>

*First a single arm without noise, started exactly at the maximum (arm stretched away from the target). Then 200 translucent arms with noise from the same pose; on the landscape, each run is a point. Last, a close-up of the cloud at the solution, with the ellipse predicted by the formula below; halfway through, the step size is halved.*

- **Without noise** the arm has not moved after 200 steps: in this computation the gradient at the start is zero to the last bit. In floating point this depends on how the gradient is coded: another coding of the same function rounds to a tiny non-zero gradient and leaves after a few dozen steps. A start displaced by $\delta$ leaves after about $\ln(0.3/\delta)/\ln(1 + \alpha\lvert\lambda_{\min}\rvert)$ steps, where $\lambda_{\min}$ is the most negative curvature there; rounding acts like a very small noise.
- **With noise** ($\sigma_g = 0.04$, $\alpha = 1$) every one of the 200 runs leaves the maximum, after a median of 5 steps, and the crowd splits: 103 runs fall into the elbow-up solution and 97 into the elbow-down one (with 20 000 runs: 50.1 % and 49.9 %).
- **The noise floor.** With a fixed step the runs never settle; they tremble around the minimum. Near $q^\ast$ the update is linear, $q_{k+1} - q^\ast = (I - \alpha H)(q_k - q^\ast) - \alpha\sigma_g\xi_k$, and its stationary variance along eigenvector $i$ of $H$ is (Lecture 4)

$$
v_i = \frac{\alpha\,\sigma_g^2}{\lambda_i\,(2 - \alpha\lambda_i)} .
$$

| | measured variance / formula (flat, steep) | tool spread (rms per axis) |
|---|---|---|
| $\alpha = 1$ | 1.012, 1.005 | 3.1 cm |
| $\alpha = 0.5$ | 1.005, 1.001 | 2.1 cm |

(4000 runs each; standard error of each ratio 0.25 %. Three ratios are within two standard errors of 1; the flat direction at $\alpha = 1$ is 1.2 % high, about five standard errors, more than sampling error: the formula is exact only for the linearised update, and this cloud is 0.1 rad wide.) Halving the step slightly more than halves the variance (ratios 0.49 and 0.42), because the factor $2 - \alpha\lambda_i$ in the denominator grows as $\alpha$ shrinks. A constant step size gives a constant floor: to converge, the step size must shrink.

**Temperature.** Write the noise as $\sqrt{2\alpha T}\,\xi_k$. This is the Langevin step of Lab 3, and for small $\alpha$ the runs are distributed with density proportional to $e^{-f/T}$. Lower minima are visited more often, more so at low temperature. To see it, add a preference that makes the two solutions unequal: the cost $f_w = \tfrac12\lVert p - t\rVert^2 - w\,z_{\text{elbow}}$ with $w = 0.04$ m rewards a high elbow. The elbow-up minimum is now the deeper one (tool 3.6 mm from the target, against 39.8 mm for elbow down).

- At a fixed temperature the share of runs in the deep well matches the prediction from $e^{-f/T}$: 0.915 at $T = 0.008$ and 0.608 at $T = 0.04$ (prediction 0.915 and 0.608; 2000 runs started in each well). At $T = 0.004$ the barrier is too high for the runs to mix within 1.5 million steps, and the prediction is not reached.
- **Annealing.** Start all runs in the shallow well and lower $T$ geometrically from 0.03 to 0.002. The share that ends in the deep well depends on how slowly you cool:

| Steps of the cooling schedule | $10^3$ | $10^4$ | $10^5$ | $10^6$ |
|---|---|---|---|---|
| share in the deep well | 10.7 % | 60.2 % | 85.3 % | 93.8 % |

Dropping straight to $T = 0.002$ leaves every run where it started (0 %). What counts is the time spent near $T \approx 0.008$ to $0.012$, warm enough to cross the barrier and cool enough to prefer the deep side.

> **Takeaway.** Noise lets gradient descent leave maxima and saddles, but with a fixed step it never settles: the cloud has the size $v_i$ predicts. Lowering the noise slowly sends the runs to the deeper minimum.
> ▶ Try it: [Gradient descent with noise](https://realgaule.github.io/rob7104/lab6/#d21): change the noise and the step size, start at the stretched pose, cool down.

## 2.2 $\ell_1$ and sparsity: moving as few joints as possible

**The problem.** The arm starts at a rest pose $q_0$ and should reach a target about 30 cm away by changing its joints by $d$. Among all ways of doing it, prefer small changes:

$$
\min_d\ \tfrac12\,\lVert p(q_0 + d) - t\rVert^2 + \lambda\,\lVert d\rVert_1
\qquad\text{or}\qquad
\min_d\ \tfrac12\,\lVert p(q_0 + d) - t\rVert^2 + \tfrac{\lambda}{2}\,\lVert d\rVert_2^2 .
$$

**Why $\ell_1$ gives zeros: the geometry.** Lecture 4's two-variable example: among all $d$ with $2d_1 + d_2 = 2$, the smallest $\ell_2$ norm is at $(0.8, 0.4)$, where a circle touches the line, but the smallest $\ell_1$ norm is at $(1, 0)$, where a corner of the diamond touches it. The corners of the $\ell_1$ ball lie on the axes, so the solution tends to have zero entries.

**Why $\ell_1$ gives zeros: the algorithm.** The kink of $\lvert d_j\rvert$ at zero has no gradient, so the local model keeps it as it is and linearises only the smooth part. That is the **proximal gradient** method (ISTA):

$$
d \leftarrow S_{\alpha\lambda}\big(d - \alpha\,\nabla g(d)\big),
\qquad
S_\tau(v) = \operatorname{sign}(v)\,\max(\lvert v\rvert - \tau,\ 0),
$$

where $g$ is the smooth reaching term. The soft threshold $S_\tau$ shifts every entry towards zero and sets the small ones to exactly zero. The $\ell_2$ penalty only scales entries, $v/(1 + \alpha\lambda)$, and never makes one zero.

<video controls loop muted playsinline width="100%" src="media/s22_l1.mp4" poster="media/s22_l1_poster.png"></video>

*A ring on a joint shows how much that joint moved; a grey ring means exactly zero. First the $\ell_2$ solution (blue), then the $\ell_1$ solution (yellow), both leaving the same 0.68 mm gap to the target. Then the $\ell_1$ weight grows and the rings go out one by one.*

| | Joints that move | Joint changes |
|---|---|---|
| $\ell_2$ penalty | 5 | 26.7°, 19.0°, −34.1°, −18.0°, 9.5°, 0° |
| $\ell_1$ penalty | 3 | 28.6°, 23.1°, −43.6°, then exactly 0, 0, 0 |

Only five joints can matter: the tool point lies on the axis of the last joint, so turning it does not move the tool point. The $\ell_1$ solution moves the three joints with the long lever arms (shoulder pan, shoulder lift, elbow).

**Why three.** A position target is three equations. For the linearised problem, minimising $\lVert d\rVert_1$ subject to $J d = \Delta p$ is a linear program, and its solution is a vertex with **at most three** non-zero entries: this is basis pursuit (Lecture 4). For the nonlinear penalised problem it holds in most cases but not all: in 10 500 test cases, more than three joints moved in 0.6 % to 3 % of them (target steps of 5 to 30 cm). From the rest pose of the video, three joints moved in 200 of 200 random targets.

**The price: shrinkage.** As $\lambda$ grows, the joints stop one at a time (the shoulder lift at $0.28\,\lambda_{\max}$, the shoulder pan at $0.79\,\lambda_{\max}$, the elbow at $\lambda_{\max} = \max_j \lvert (J^\top(p - t))_j\rvert$ at the start), and nothing moves above $\lambda_{\max}$. For small $\lambda$ the gap to the target grows in proportion to $\lambda$: sparsity costs accuracy.

The same mechanism is behind compressed sensing in Lecture 4: basis pursuit recovers a sparse signal exactly from few measurements (200 of 200 random instances), while the minimum-$\ell_2$ solution misses it by a relative error of about 0.79.

> **Takeaway.** An $\ell_1$ penalty has corners, and solutions sit on corners: few joints move, the others stay exactly still. The soft threshold is the algorithm that produces the exact zeros.
> ▶ Try it: [$\ell_1$ and sparsity](https://realgaule.github.io/rob7104/lab6/#d22): slide the weight, switch between $\ell_1$ and $\ell_2$, move the target.

---

# Part 3 — When the unknown is not a vector

## 3.1 Calculus of variations: optimizing a whole path

**The problem.** The hand should travel from $A$ to $B$ in time $T$ along a path $y(t)$ in a horizontal plane, without wasting effort and keeping away from a pillar. The unknown is a whole function:

$$
J[y] = \int_0^T \Big(\tfrac12\,\lVert y'(t)\rVert^2 + U\big(y(t)\big)\Big)\,dt,
\qquad y(0) = A,\quad y(T) = B .
$$

$U$ is a smooth penalty around the pillar (a bump), not a hard wall.

**The first variation.** Perturb the path, $y + \varepsilon\eta$, and differentiate at $\varepsilon = 0$. Integrating by parts,

$$
\delta J = \int_0^T \big(y'\cdot\eta' + \nabla U\cdot\eta\big)\,dt
= \big[\,y'\cdot\eta\,\big]_0^T + \int_0^T \big(-y'' + \nabla U(y)\big)\cdot\eta\,dt .
$$

With both ends fixed, $\eta(0) = \eta(T) = 0$, and $\delta J = 0$ for every $\eta$ gives the **Euler–Lagrange equation**

$$
y'' = \nabla U(y).
$$

**Discretise.** Replace the path by $M + 1$ beads $y_0 = A, \dots, y_M = B$ with spacing $h = T/M$:

$$
J_h = \sum_{j} \frac{1}{2h}\lVert y_{j+1} - y_j\rVert^2 + h\sum_j U(y_j).
$$

Now it is an ordinary optimization over the beads, and everything from Part 1 applies. The Hessian of the first sum is $\tfrac1h$ times the tridiagonal matrix with $2$ on the diagonal and $-1$ beside it. Its eigenvectors are sine waves along the path, with eigenvalues $\tfrac{4}{h}\sin^2\!\big(\tfrac{j\pi}{2M}\big)$, so its condition number $\cot^2\!\big(\tfrac{\pi}{2M}\big) \approx (2M/\pi)^2$ grows with the square of the number of beads.

<video controls loop muted playsinline width="100%" src="media/s31_variational.mp4" poster="media/s31_variational_poster.png"></video>

*A bent, hand-drawn path relaxes by gradient descent (time-lapse), beads coloured by the Euler–Lagrange residual. Then safeguarded Newton from the same start, and the arm follows the optimal path. Last, the end point may be anywhere on a shelf.*

**Gradient descent** with $\alpha = 1/L$ multiplies each sine wave of the error by $1 - \alpha\tfrac{4}{h}\sin^2\!\big(\tfrac{j\pi}{2M}\big)$ per step. The fine wrinkles shrink by a factor 0.63 per step and are gone almost at once; the large bend shrinks by 0.999 per step and takes over a thousand. Doubling the number of beads multiplies the work by about 4:

| Number of segments $M$ | 20 | 40 | 80 | 160 |
|---|---|---|---|---|
| gradient descent: steps until every bead is within 1 mm | 419 | 1513 | 5892 | 23 406 |
| safeguarded Newton: iterations | 4 | 4 | 4 | 6 |

The Hessian is banded, so a Newton step costs only about as much as a few gradient steps. Without the pillar the cost is quadratic and one Newton solve gives the exact optimum. With the pillar the problem is not convex: the optimal path passes on the robot's side, 0.19 m from the pillar's axis (cost 0.408), but a path around the far side is also a local minimum (cost 0.461), and pure Newton from the hand-drawn start ends there.

**A free end.** Let the end point be any point on a shelf (a line with direction $\tau$), so $\eta(T) = s\,\tau$ is no longer zero. The boundary term of $\delta J$ must vanish by itself, which gives the **natural boundary condition**

$$
y'(T)\cdot\tau = 0 :
$$

the optimal path meets the shelf at a right angle. In the video the fixed-end path meets the shelf at 52°; with a free end the end slides 27 cm along the shelf and the path arrives at 90.0°. The right angle needs the conditions of this derivation: fixed travel time, no cost on the end point, the plain energy $\tfrac12\lVert y'\rVert^2$, and an end that is not at the edge of the shelf. Weighting the two directions of motion differently gives 68°, and adding a cost on the end point gives 113°.

> **Takeaway.** A path is a vector with many entries. Its Euler–Lagrange equation is the condition "gradient = 0", and the finer the discretisation, the worse gradient descent is conditioned, while Newton's method does not care.
> ▶ Try it: [Calculus of variations](https://realgaule.github.io/rob7104/lab6/#d31): relax the path, move the pillar, free the end.

## 3.2 Optimization on manifolds

**The problem.** A camera on the tool should look at a target. The viewing direction is a unit vector $x$ with $x^\top x = 1$: a point on the sphere. The full orientation of the tool is a rotation matrix $X = [x_1\ x_2\ x_3]$ with $X^\top X = I$ and $\det X = 1$. These sets are **manifolds**: curved surfaces inside a vector space. Maximise the alignment with the target direction $t$ or the target rotation $R_t$:

$$
f(x) = -\,t^\top x \quad (x^\top x = 1),
\qquad
f(X) = -\operatorname{tr}\big(R_t^\top X\big) \quad (X^\top X = I).
$$

**The plain step leaves the manifold.** $x - \alpha\nabla f(x) = x + \alpha t$ is no longer a unit vector. At the solution the ordinary gradient is not even zero: it is normal to the sphere, $\nabla f = \mu\,x$, the Lagrange condition of Lecture 5. Only its tangential part matters.

**The local model on a manifold.** Step in the tangent space, then pull back (Lecture 4):

$$
\text{sphere:}\quad x \leftarrow \frac{x - \alpha\,(I - xx^\top)\nabla f}{\big\lVert x - \alpha\,(I - xx^\top)\nabla f\big\rVert},
\qquad
\text{rotations:}\quad X \leftarrow \operatorname{qr}\!\Big(X - \alpha\big(G - X\operatorname{sym}(X^\top G)\big)\Big),
$$

with $G = \nabla f(X)$, $\operatorname{sym}(A) = \tfrac12(A + A^\top)$, and $\operatorname{qr}$ the orthogonal factor of the QR decomposition. $I - xx^\top$ and $G - X\operatorname{sym}(X^\top G)$ project onto the tangent space; normalising and the QR factor are **retractions**, maps that bring a point near the manifold back onto it.

<video controls loop muted playsinline width="100%" src="media/s32_manifold.mp4" poster="media/s32_manifold_poster.png"></video>

*First the viewing direction on the sphere, then the whole gripper orientation. Red: the plain gradient step; white: tangent step plus retraction. Bottom right: the constraint residual against the iteration, on a logarithmic axis.*

With $\alpha = 0.6$:

| | Plain gradient step | Tangent step + retraction |
|---|---|---|
| sphere | $\lVert x\rVert$ = 1.24, 1.67, 2.18 after 1, 2, 3 steps; residual grows | every iterate a unit vector (residual below $4\times10^{-16}$); from 82° to under 1° in 6 steps |
| rotations | after 3 steps singular values 2.8, 1.93, 1.93 and determinant 10.4: the gripper is stretched and sheared | every iterate a rotation (residual about $10^{-15}$, determinant 1); from 98° to under 1° in 7 steps |

In the video the plain step visibly deforms the gripper: a matrix that is not a rotation applied to a rigid body. A tangent step without the retraction also leaves the manifold, only more slowly (to second order in the step); the pull-back is what keeps every iterate admissible.

> **Takeaway.** On a curved set, take the step in the tangent space and retract. Every iterate then satisfies the constraint, instead of only the limit.
> ▶ Try it: [Optimization on manifolds](https://realgaule.github.io/rob7104/lab6/#d32): move the target and switch the retraction on and off.

---

## 4. Summary

Every method in this lab builds a local model at the current point and steps to its minimum.

| Structure of the problem | Local model | Method | What to watch | Seen on |
|---|---|---|---|---|
| smooth, slope only | tangent plane + $\tfrac1{2\alpha}\lVert d\rVert^2$ | gradient descent | $\alpha < 2/L$; condition number | 1.1: step size, reach limit |
| smooth, curvature known | quadratic | Newton (with safeguard) | is $H$ positive definite? | 1.2: basin map |
| sum of squared residuals | linearised residual | Gauss–Newton, damped least squares | smallest singular value of $J$ | 1.3: inverse kinematics |
| noisy gradient | slope + noise | noisy gradient descent, annealing | noise floor $v_i$; cooling speed | 2.1: crowd of arms |
| non-smooth ($\ell_1$) | linearised smooth part + exact kink | proximal gradient | which entries are exactly zero; shrinkage | 2.2: rings on the joints |
| unknown is a function | first variation | Euler–Lagrange, discretised | conditioning grows with the grid; boundary conditions | 3.1: path around a pillar |
| unknown on a manifold | tangent step + retraction | Riemannian gradient descent | constraint residual | 3.2: camera orientation |

**What this lab left out.** Quasi-Newton methods such as BFGS build the quadratic model of 1.2 from gradients alone, without computing $H$. Adam rescales noisy gradients per coordinate, as in machine learning. Inequality constraints and the KKT conditions extend the Lagrange condition of 3.2.

**Notes on the simulations.**

- The arm is the UR5e model of MuJoCo Menagerie, with the model's rounded link lengths. It is moved kinematically: the joint angles are set to the optimizer's iterates. There is no simulation of forces, friction or contact; the pillar acts only through the penalty $U$ of 3.1.
- Joint limits are ignored and angles are periodic, so some poses in the landscapes would collide with the arm itself or with its base.
- Step counts are always quoted with their stopping rule (within 1 mm of the target, or $\lVert\nabla f\rVert \le 10^{-9}$). Random experiments state their number of runs. All numbers come from the scripts that made the videos and were checked by an independent implementation.